In [ ]:
from ndtools import staged_max_flow as smf
from typing import Any, Dict, List, Sequence

from pathlib import Path
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from rsr import rsr
import torch

import time

# Utility functions

For use in plotting later.

In [ ]:
def relabel_selected_keys(
    selected_keys: Sequence[str],
    nodes: Dict[str, Dict[str, Any]],
    edges: Dict[str, Dict[str, Any]],
    *,
    node_label: str = "node_id",
    edge_label_fmt: str = "({from_}, {to})",
    keep_unmatched: bool = True,
) -> List[str]:
    """
    Given a list of comp_id keys (e.g., ['x1','x29',...]), return a same-length list
    where each key is replaced by:
      - node label (default: node id like 'n1') if comp_id matches a node
      - '(from,to)' if comp_id matches an edge
    First match wins if multiple components share a comp_id.
    Node takes precedence over edge if both match.
    """

    comp_to_label: Dict[str, str] = {}

    for n_id, n in nodes.items():
        comp_id = n.get("comp_id")
        if comp_id and comp_id not in comp_to_label:
            comp_to_label[comp_id] = n_id if node_label == "node_id" else str(n.get(node_label, n_id))

    for _, e in edges.items():
        comp_id = e.get("comp_id")
        if comp_id and comp_id not in comp_to_label:
            comp_to_label[comp_id] = edge_label_fmt.format(from_=e.get("from"), to=e.get("to"))

    out_keys: List[str] = []
    for k in selected_keys:
        if k in comp_to_label:
            out_keys.append(comp_to_label[k])
        else:
            if keep_unmatched:
                out_keys.append(k)
            else:
                continue

    return out_keys

# System reliability

## Case 1: Original system

First load data.

In [ ]:
DATASET = Path(r"data")

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs.json").read_text(encoding="utf-8"))

In [ ]:
def s_fun(comps_st):
    flow, sys_st_str, min_comp_state = smf.sys_fun( comps_st, nodes, edges, probs_dict, target_flow = 90.0 )

    sys_st = 1 if sys_st_str == 's' else 0
    return flow, sys_st, None

row_names = list(probs_dict.keys())
n_state = 2  # binary states: 0, 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

In [ ]:
RSRPATH = Path("rsr_res")

sys_upper_st = 1  # system survival state

refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
refs_mat_upper = refs_mat_upper.to(device)
refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
refs_mat_lower = refs_mat_lower.to(device)

## System marginal probability

In [ ]:
st = time.time()

pr_cond = rsr.get_comp_cond_sys_prob(
    refs_mat_upper,
    refs_mat_lower,
    probs,
    comps_st_cond = {},
    row_names = row_names,
    s_fun = s_fun,
    sys_upper_st = sys_upper_st,
    n_sample = 10_000_000
)

et = time.time()
print(f"Time elapsed: {et - st:.2f} seconds\n")
print(f"P(sys >= {sys_upper_st}) = {pr_cond['upper']:.3e}")
print(f"P(sys <= {sys_upper_st-1} ) = {pr_cond['lower']:.3e}\n")

# Component Importance Measures

For each component $X$ we compute three importance measures:

- **Birnbaum's measure (BI)**: $BI(x) = P(S=1 \mid X=1) - P(S=1 \mid X=0)$
- **Risk Achievement Worth (RAW)**: $RAW(x) = \dfrac{P(S=0 \mid X=1)}{P(S=0)}$
- **Risk Reduction Worth (RRW)**: $RRW(x) = \dfrac{P(S=0 \mid X=0)}{P(S=0)}$

All three derive from the same two conditional probabilities, so they are computed together to avoid duplicate sampling.

In [ ]:
bi_dict = {}
raw_dict = {}
rrw_dict = {}

p_sys_fail = pr_cond['lower']
p_sys_surv = pr_cond['upper']

for x in probs_dict.keys():
    print(f"Calculating measures for {x}..")

    ps1_x1 = rsr.get_comp_cond_sys_prob(
        refs_mat_upper,
        refs_mat_lower,
        probs,
        comps_st_cond = {x: 1},
        row_names = row_names,
        s_fun = s_fun,
        sys_upper_st = sys_upper_st,
        n_sample = 10_000_000
    )

    ps1_x0 = rsr.get_comp_cond_sys_prob(
        refs_mat_upper,
        refs_mat_lower,
        probs,
        comps_st_cond = {x: 0},
        row_names = row_names,
        s_fun = s_fun,
        sys_upper_st = sys_upper_st,
        n_sample = 10_000_000
    )

    p_surv_x1 = ps1_x1['upper']
    p_surv_x0 = ps1_x0['upper']
    p_fail_x1 = ps1_x1['lower']
    p_fail_x0 = ps1_x0['lower']

    # Birnbaum's measure
    bi = max(0.0, p_surv_x1 - p_surv_x0)

    # Risk Achievement Worth: P(S=0|X=1) / P(S=0)
    raw = p_fail_x1 / p_sys_fail if p_sys_fail > 0 else float('inf')

    # Risk Reduction Worth: P(S=0|X=0) / P(S=0)
    rrw = p_fail_x0 / p_sys_fail if p_sys_fail > 0 else float('inf')

    bi_dict[x] = bi
    raw_dict[x] = raw
    rrw_dict[x] = rrw

    print(f"BI({x}) = {bi:.3e}, RAW({x}) = {raw:.3e}, RRW({x}) = {rrw:.3e}\n")

results_dict = {
    "P_sys_surv": p_sys_surv,
    "P_sys_fail": p_sys_fail,
    "BI": bi_dict,
    "RAW": raw_dict,
    "RRW": rrw_dict,
}

out_path = RSRPATH / "importance_measures.json"
with open(out_path, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"Results saved to {out_path}")

## Comparison plot

BI is bounded in $[0, 1]$ while RAW and RRW are ratios that may take large values, so the three measures are shown as separate panels. Each panel ranks the top 10 components by that measure.

In [ ]:
"""
out_path = RSRPATH / "importance_measures.json"
with open(out_path, "r") as f:
    results_dict = json.load(f)
bi_dict = results_dict["BI"]
raw_dict = results_dict["RAW"]
rrw_dict = results_dict["RRW"]
"""

In [ ]:
n_data_vis = 10  # number of components to visualise per measure
FONT_FAMILY = "Times New Roman"
LABEL_SIZE = 16
TICK_SIZE = 14
TITLE_SIZE = 18

FIGPATH = "figs"
Path(FIGPATH).mkdir(parents=True, exist_ok=True)

mpl.rcParams["font.family"] = FONT_FAMILY

# (title, dict, descending?)
# RAW = P(S=0|X=1)/P(S=0): lower => more important, so sort ascending.
measures = [
    ("Birnbaum's measure", bi_dict, True),
    ("Risk Achievement Worth", raw_dict, False),
    ("Risk Reduction Worth", rrw_dict, True),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, (title, mdict, descending) in zip(axes, measures):
    finite_items = [(k, v) for k, v in mdict.items() if np.isfinite(v)]
    sorted_items = sorted(finite_items, key=lambda kv: kv[1], reverse=descending)
    top_items = sorted_items[:n_data_vis]
    keys, values = zip(*top_items)
    keys_rlb = relabel_selected_keys(list(keys), nodes, edges)

    ax.bar(keys_rlb, values)
    ax.set_xlabel("Top 10 components", fontsize=LABEL_SIZE)
    ax.set_ylabel(title, fontsize=LABEL_SIZE)
    ax.set_title(title, fontsize=TITLE_SIZE)
    ax.tick_params(axis='x', rotation=45, labelsize=TICK_SIZE)
    ax.tick_params(axis='y', labelsize=TICK_SIZE)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')

plt.tight_layout()
out_fig = Path(FIGPATH) / "importance_measures_top10.png"
plt.savefig(out_fig, dpi=300)
plt.show()